# Тренування, валідація та порівняння моделей

Мета notebook: навчити кілька регресійних моделей для прогнозування `latency` та порівняти їх за метриками MAE, RMSE і MAPE.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "hwnasbench_nasbench201.csv"
DATASET_PATH

WindowsPath('P:/pythonProject5/data/processed/hwnasbench_nasbench201.csv')

## Завантаження та підготовка даних

In [2]:
df = pd.read_csv(DATASET_PATH)
df = df[df["latency"] > 0].copy()

df.shape

(267153, 17)

In [3]:
CATEGORICAL_FEATURES = ["dataset", "device"]
NUMERIC_FEATURES = [
    "base_channels",
    "num_cells",
    "num_classes",
    "op_count_avg_pool_3x3",
    "op_count_nor_conv_1x1",
    "op_count_nor_conv_3x3",
    "op_count_skip_connect",
    "op_count_none",
]
TARGET = "latency"

x = df[CATEGORICAL_FEATURES + NUMERIC_FEATURES]
y = df[TARGET]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
)

x_train.shape, x_test.shape

((213722, 10), (53431, 10))

## Допоміжні функції

In [4]:
def mape(y_true, y_pred):
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)


def build_pipeline(model):
    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
            ("numeric", "passthrough", NUMERIC_FEATURES),
        ]
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


def build_scaled_pipeline(model):
    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
            ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ]
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


def evaluate(model_name, pipeline):
    predictions = pipeline.predict(x_test)
    return {
        "model": model_name,
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
        "mape_percent": mape(y_test, predictions),
    }

## Моделі для порівняння

Відповідно до ТЗ порівнюються три підходи:

1. лінійна регресія як baseline;
2. gradient boosting;
3. нейронна мережа MLP.

In [5]:
models = {
    "linear_regression": build_pipeline(LinearRegression()),
    "gradient_boosting": build_pipeline(
        HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=42,
        )
    ),
    "neural_network_mlp": build_scaled_pipeline(
        MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            learning_rate_init=0.001,
            batch_size=512,
            max_iter=120,
            early_stopping=True,
            random_state=42,
        )
    ),
}

## Навчання та оцінка

In [6]:
results = []
trained_models = {}

for model_name, pipeline in models.items():
    pipeline.fit(x_train, y_train)
    trained_models[model_name] = pipeline
    results.append(evaluate(model_name, pipeline))

results_df = pd.DataFrame(results).sort_values("mae")
results_df

,model,mae,rmse,mape_percent
1,gradient_boosting,0.874645,2.158862,187.250016
2,neural_network_mlp,0.875913,2.156725,184.209514
0,linear_regression,3.979043,6.043252,455.166435


## ?????? ??????????? ?????? ???????

????? ????????? ??????? ?????? ????????? ??????????, ???? ?????? ???? ????? ?????????? ? ????????? ??????? ??????? ??? ????.

In [7]:
from IPython.display import Markdown, display

best_row = results_df.iloc[0]
best_mae = float(best_row["mae"])
linear_mae = float(results_df.loc[results_df["model"] == "linear_regression", "mae"].iloc[0])
gb_mae = float(results_df.loc[results_df["model"] == "gradient_boosting", "mae"].iloc[0])
mlp_mae = float(results_df.loc[results_df["model"] == "neural_network_mlp", "mae"].iloc[0])

gb_mlp_gap = abs(gb_mae - mlp_mae)
gb_mlp_gap_percent = gb_mlp_gap / min(gb_mae, mlp_mae) * 100
linear_to_best_ratio = linear_mae / best_mae

model_comparison_analysis = pd.DataFrame(
    [
        {"analysis_item": "???????? ?????? ?? MAE", "value": best_row["model"]},
        {"analysis_item": "MAE ????????? ??????", "value": f"{best_mae:.4f}"},
        {
            "analysis_item": "? ??????? ????? Linear Regression ????? ?? ????????",
            "value": f"{linear_to_best_ratio:.2f}x",
        },
        {
            "analysis_item": "??????? MAE ??? Gradient Boosting ? MLP",
            "value": f"{gb_mlp_gap:.4f} ({gb_mlp_gap_percent:.2f}%)",
        },
    ]
)

display(model_comparison_analysis)

display(
    Markdown(
        f"""
**????????????? ???????????.**

`Linear Regression` ??? ?????? ?????? ???????, ???? ?? latency ?? ?????????? ??????? ???????? ?????????? ??? ????????? ???????? ? ?????????????? ?????. ????? ??????????? ???????? ??? ????????, dataset ?? ????????? ?????, ????????? `device ? op_count_nor_conv_3x3`.

`Gradient Boosting` ? `MLP` ???????? ?????? ?????, ?? ?????? ?????? ?????? ????????? ????????? ??????????. ??????? ??? ???? ??????????: `{gb_mlp_gap:.4f}` MAE, ????? ????????? `{gb_mlp_gap_percent:.2f}%`. ???? MLP ????????? ? ????????? ?? MAE, ??? Gradient Boosting ????????? ?? ???????????? ? ??????????? ??????? ????????????? ??? ????????? ?????.
"""
    )
)

,analysis_item,value
0,???????? ?????? ?? MAE,gradient_boosting
1,MAE ????????? ??????,0.8746
2,? ??????? ????? Linear Regression ????? ?? ???...,4.55x
3,??????? MAE ??? Gradient Boosting ? MLP,0.0013 (0.14%)



**????????????? ???????????.**

`Linear Regression` ??? ?????? ?????? ???????, ???? ?? latency ?? ?????????? ??????? ???????? ?????????? ??? ????????? ???????? ? ?????????????? ?????. ????? ??????????? ???????? ??? ????????, dataset ?? ????????? ?????, ????????? `device ? op_count_nor_conv_3x3`.

`Gradient Boosting` ? `MLP` ???????? ?????? ?????, ?? ?????? ?????? ?????? ????????? ????????? ??????????. ??????? ??? ???? ??????????: `0.0013` MAE, ????? ????????? `0.14%`. ???? MLP ????????? ? ????????? ?? MAE, ??? Gradient Boosting ????????? ?? ???????????? ? ??????????? ??????? ????????????? ??? ????????? ?????.


## Найкраща модель

In [8]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]

best_model_name

'gradient_boosting'

## Приклади прогнозів

In [9]:
sample = x_test.head(10).copy()
sample["actual_latency"] = y_test.head(10).values
sample["predicted_latency"] = best_model.predict(x_test.head(10))
sample["absolute_error"] = (sample["actual_latency"] - sample["predicted_latency"]).abs()

sample

,dataset,device,base_channels,num_cells,num_classes,op_count_avg_pool_3x3,op_count_nor_conv_1x1,op_count_nor_conv_3x3,op_count_skip_connect,op_count_none,actual_latency,predicted_latency,absolute_error
81292,cifar10,eyeriss,16,5,10,2,0,3,1,0,9.052340,8.793393,0.258947
194928,ImageNet16-120,edgegpu,16,5,120,1,1,1,3,0,5.585508,5.414766,0.170742
227298,ImageNet16-120,edgegpu,16,5,120,2,1,1,1,1,6.195378,5.272232,0.923146
98431,cifar100,raspi4,16,5,100,2,1,1,0,2,27.237978,23.850378,3.387600
163042,cifar100,eyeriss,16,5,100,2,1,0,1,2,4.877440,4.755287,0.122153
44217,cifar10,pixel3,16,5,10,2,0,2,1,1,13.981100,16.209333,2.228233
140762,cifar100,edgetpu,16,5,100,0,0,0,4,2,0.438502,0.310038,0.128464
153487,cifar100,raspi4,16,5,100,1,2,0,1,2,14.541051,11.268691,3.272360
80738,cifar10,edgetpu,16,5,10,2,0,1,2,1,1.304473,1.194468,0.110005
263258,ImageNet16-120,edgetpu,16,5,120,1,1,3,0,1,1.017552,1.133742,0.116191


## ?????? ????????? ?????????

????? ?????? 10 ?????????, ?????????? ???????? ??????? ?? ???? ???????? ??????? ?? ?'??????, ?? ???? ?????? ?????? ??????????? ?????????.

In [10]:
prediction_analysis = x_test.copy()
prediction_analysis["actual_latency"] = y_test.values
prediction_analysis["predicted_latency"] = best_model.predict(x_test)
prediction_analysis["error"] = prediction_analysis["predicted_latency"] - prediction_analysis["actual_latency"]
prediction_analysis["absolute_error"] = prediction_analysis["error"].abs()
prediction_analysis["absolute_percentage_error"] = (
    prediction_analysis["absolute_error"] / prediction_analysis["actual_latency"] * 100
)

error_quantiles = prediction_analysis["absolute_error"].quantile([0.50, 0.75, 0.90, 0.95, 0.99])
error_quantiles_df = error_quantiles.reset_index()
error_quantiles_df.columns = ["quantile", "absolute_error"]

error_by_device = (
    prediction_analysis.groupby("device")
    .agg(
        rows=("absolute_error", "size"),
        mean_actual_latency=("actual_latency", "mean"),
        mae=("absolute_error", "mean"),
        max_absolute_error=("absolute_error", "max"),
    )
    .sort_values("mae", ascending=False)
    .reset_index()
)

worst_predictions = prediction_analysis.sort_values("absolute_error", ascending=False).head(10)

display(Markdown("### ???????? ?????????? ???????"))
display(error_quantiles_df)

display(Markdown("### ??????? ?? ??????????"))
display(error_by_device)

display(Markdown("### ????????? ??????????"))
display(
    worst_predictions[
        [
            "dataset",
            "device",
            "actual_latency",
            "predicted_latency",
            "error",
            "absolute_error",
            "absolute_percentage_error",
        ]
    ]
)

display(
    Markdown(
        f"""
**???????? ?? ?????????? ?????????.**

???????? ????????? ??????? ????????? `{error_quantiles.loc[0.50]:.4f}`, ????? ??? ????????? ???????? ??????? ??????? ???????? ?? ?????????? ????????. ???????? 95-? ?????????? ???????? `{error_quantiles.loc[0.95]:.4f}`, ?? ??????? ????????? ??????? ???????? ????????.

????????? ??????? ??????? ??????????????? ??? ???????? `{error_by_device.iloc[0]['device']}`. ?? ???????, ?? ?????? ????????????? ???????? ??? hardware-?????????: ?????? ????? ?????? ?? ????????? ?? ????????? ? ???????????? ?????????? latency, ??? ????? ???????????? ?? ?????????? ??? ????? ?????????? ????????.
"""
    )
)

### ???????? ?????????? ???????

,quantile,absolute_error
0,0.50,0.207941
1,0.75,0.875447
2,0.90,2.300799
3,0.95,3.624591
4,0.99,8.421053


### ??????? ?? ??????????

,device,rows,mean_actual_latency,mae,max_absolute_error
0,raspi4,9216,20.239806,2.724250,34.316240
1,pixel3,9409,9.185630,1.289284,18.787438
2,edgegpu,9372,5.044619,0.705033,4.609143
3,edgetpu,6582,1.042647,0.118967,0.888799
4,eyeriss,9533,4.117348,0.117932,1.143640
5,fpga,9319,2.864516,0.105241,0.596829


### ????????? ??????????

,dataset,device,actual_latency,predicted_latency,error,absolute_error,absolute_percentage_error
83377,cifar10,raspi4,1.227833,35.544074,34.316240,34.316240,2794.861281
34537,cifar10,raspi4,4.923285,36.206684,31.283399,31.283399,635.417201
1921,cifar10,raspi4,10.027026,41.175033,31.148006,31.148006,310.640517
59533,cifar10,raspi4,2.807009,33.819889,31.012880,31.012880,1104.837226
40549,cifar10,raspi4,5.648446,36.438664,30.790218,30.790218,545.109554
97387,cifar100,raspi4,5.295944,36.047947,30.752003,30.752003,580.670835
25477,cifar10,raspi4,8.033053,38.744764,30.711711,30.711711,382.316804
23839,cifar10,raspi4,2.896928,33.219524,30.322596,30.322596,1046.715729
113659,cifar100,raspi4,7.767616,37.973138,30.205522,30.205522,388.864783
117493,cifar100,raspi4,7.801699,37.973138,30.171439,30.171439,386.729063



**???????? ?? ?????????? ?????????.**

???????? ????????? ??????? ????????? `0.2079`, ????? ??? ????????? ???????? ??????? ??????? ???????? ?? ?????????? ????????. ???????? 95-? ?????????? ???????? `3.6246`, ?? ??????? ????????? ??????? ???????? ????????.

????????? ??????? ??????? ??????????????? ??? ???????? `raspi4`. ?? ???????, ?? ?????? ????????????? ???????? ??? hardware-?????????: ?????? ????? ?????? ?? ????????? ?? ????????? ? ???????????? ?????????? latency, ??? ????? ???????????? ?? ?????????? ??? ????? ?????????? ????????.


## Висновок

У цьому експерименті побудовано й порівняно три регресійні підходи для прогнозування inference latency. Найкраща модель визначається за мінімальним MAE. MAPE наведено як додаткову метрику, але для цього датасету вона менш стабільна через наявність дуже малих latency-значень.